In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the data
raw = pd.read_csv('gcdg_zaf.csv', low_memory=False)
print(raw.shape)
print(raw['subjid'].nunique(), 'children')

(4176, 363)
2389 children


In [3]:
print(raw['agedays'].describe())
print('missing age:', raw['agedays'].isna().sum())

count    4175.000000
mean      943.178683
std       450.780117
min       125.000000
25%       721.000000
50%       782.000000
75%      1461.000000
max      1461.000000
Name: agedays, dtype: float64
missing age: 1


In [4]:
df = raw[raw['agedays'] <= 3 * 365].copy()
df['age_months'] = df['agedays'] / 30.4375
print('rows:', len(df))
print('children:', df['subjid'].nunique())
print(df['age_months'].min(), df['age_months'].max())

rows: 2561
children: 2033
4.1067761806981515 29.305954825462013


In [5]:
# Checking Data Quality
meta_cols = ['ctrycd', 'cohort', 'cohortn', 'subjid', 'agedays', 'sex', 'gagebrth']
item_cols = [col for col in df.columns if col not in meta_cols + ['age_months']]
print(len(item_cols), 'item columns')
print(df[meta_cols].isna().sum())

356 item columns
ctrycd       0
cohort       0
cohortn      0
subjid       0
agedays      0
sex          0
gagebrth    53
dtype: int64


In [6]:
print('duplicate rows:', df.duplicated().sum())
print('duplicate child + age:', df.duplicated(['subjid', 'agedays']).sum())

duplicate rows: 0
duplicate child + age: 0


In [7]:
values = pd.unique(df[item_cols].values.ravel())
print(sorted(v for v in values if v == v))

[np.float64(0.0), np.float64(1.0)]


In [8]:
print(df['sex'].value_counts())
print('children with more than one sex:', (df.groupby('subjid')['sex'].nunique() > 1).sum())
print('children with more than one gestational age:', (df.groupby('subjid')['gagebrth'].nunique() > 1).sum())

sex
Female    1312
Male      1249
Name: count, dtype: int64
children with more than one sex: 0
children with more than one gestational age: 0


In [9]:
answered = df[item_cols].notna().sum(axis=1)
print(answered.describe())
print('rows with no items answered:', (answered == 0).sum())

count    2561.000000
mean       72.866458
std        69.354925
min         8.000000
25%        28.000000
50%        29.000000
75%       166.000000
max       201.000000
dtype: float64
rows with no items answered: 0


In [10]:
never_answered = [col for col in item_cols if df[col].isna().all()]
print(len(never_answered), 'items never answered:', never_answered)

21 items never answered: ['vinxxc031', 'vinxxc032', 'vinxxc033', 'vinxxc034', 'vinxxc035', 'vinxxc036', 'vinxxc037', 'vinxxc038', 'vinxxc039', 'vinxxc040', 'vinxxc041', 'vinxxc042', 'vinxxc043', 'vinxxc044', 'vinxxc045', 'vinxxc046', 'vinxxc047', 'vinxxc048', 'vinxxc049', 'vinxxc050', 'vinxxc051']


In [11]:
vineland = [col for col in item_cols if col.startswith('vin')]
others = [col for col in item_cols if not col.startswith('vin')]
has_vineland = df[vineland].notna().any(axis=1)
has_others = df[others].notna().any(axis=1)
print('Vineland only:', (has_vineland & ~has_others).sum())
print('other instruments only:', (~has_vineland & has_others).sum())
print('both:', (has_vineland & has_others).sum())

Vineland only: 1801
other instruments only: 760
both: 0


In [12]:
print(df.groupby('subjid').size().value_counts())
print(pd.cut(df['age_months'], [0, 6, 12, 18, 24, 30]).value_counts().sort_index())

1    1551
2     436
3      46
Name: count, dtype: int64
age_months
(0, 6]       216
(6, 12]      274
(12, 18]     270
(18, 24]     526
(24, 30]    1275
Name: count, dtype: int64


In [13]:
item_cols = [col for col in item_cols if col not in never_answered]
print(len(item_cols), 'items kept')

335 items kept


**Decisions**

- One row has no age and was already removed by the age filter.
- `gagebrth` is missing for some rows. It is not used for the domain scores, so no action.
- No duplicate rows, no duplicate child and age, and all item values are 0 or 1.
- Sex and gestational age are consistent within each child.
- Missing item values mean the item was not administered, not that it was failed. They are left as missing and ignored when computing domain scores.
- Items that were never answered in ages 0-3 are dropped.
- Each row comes from one instrument family only, so most rows will have some domains missing (see issue 7).
- Many children have more than one visit, so the train and test split must be by child (issue 8).
- Few records are under 12 months, which affects the age bands (issue 5).